<a href="https://colab.research.google.com/github/muhsina419/Machine_learning_-_Parallel_computing_s6/blob/main/Q6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


Question 6: Collaborative Filtering and Recommender Systems:

The objective of this lab exercise is to understand and implement collaborative filtering techniques for building recommender systems.
A. Understanding Collaborative Filtering:

    a. Provide an overview of collaborative filtering and its importance in building recommender systems.
    b. Explain the concepts of user-item interactions, user-based collaborative filtering, and item-based collaborative filtering.

B. Dataset Exploration:

    a. Select a suitable dataset for building a recommender system (e.g., movie ratings, book reviews, product ratings).
    b. Load the dataset and explore its structure and attributes.

C. User-Based Collaborative Filtering:

    a. Implement a user-based collaborative filtering algorithm to recommend items to users based on similarities between users.
    b. Discuss different similarity metrics such as cosine similarity, Pearson correlation, and Euclidean distance.
    c. Evaluate the performance of the user-based collaborative filtering approach using appropriate evaluation metrics (e.g., precision, recall, F1-score).

D. Item-Based Collaborative Filtering:

    a. Implement an item-based collaborative filtering algorithm to recommend items to users based on similarities between items.
    b. Discuss the advantages and disadvantages of item-based collaborative filtering compared to user-based collaborative filtering.
    c. Evaluate the performance of the item-based collaborative filtering approach using similar evaluation metrics as in the user-based approach.

E. Hybrid Approaches:

    a. Discuss hybrid recommender systems that combine collaborative filtering with other techniques such as content-based filtering or matrix factorization.
    b. Implement a simple hybrid recommender system by combining user-based and item-based collaborative filtering approaches.
    c. Evaluate the performance of the hybrid recommender system and compare it with the individual approaches.

F. Evaluation and Interpretation:

    a. Analyze the results of the collaborative filtering and hybrid approaches.
    b. Interpret the recommended items and their relevance to users.
    c. Discuss potential improvements and future directions for enhancing the recommender system's performance.

https://www.kaggle.com/datasets/residentmario/ramen-ratings?select=ramen-ratings.csv


In [2]:
import pandas as pd
import numpy as np
import kagglehub

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import precision_score, recall_score, f1_score

In [3]:
# Download dataset
path = kagglehub.dataset_download("residentmario/ramen-ratings")

# Load CSV file
df = pd.read_csv(f"{path}/ramen-ratings.csv")

print("\nDataset Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

Using Colab cache for faster access to the 'ramen-ratings' dataset.

Dataset Shape: (2580, 7)

Columns: ['Review #', 'Brand', 'Variety', 'Style', 'Country', 'Stars', 'Top Ten']


In [8]:
# Remove unrated entries
df = df[df['Stars'] != "Unrated"]

# Convert stars to float
df['Stars'] = df['Stars'].astype(float)

# Create User ID (Using country as user proxy)
df['User_ID'] = df['Country']

print("\nSample Data:")
print(df.head())


Sample Data:
   Review #           Brand  \
0      2580       New Touch   
1      2579        Just Way   
2      2578          Nissin   
3      2577         Wei Lih   
4      2576  Ching's Secret   

                                             Variety Style Country  Stars  \
0                          T's Restaurant Tantanmen    Cup   Japan   3.75   
1  Noodles Spicy Hot Sesame Spicy Hot Sesame Guan...  Pack  Taiwan   1.00   
2                      Cup Noodles Chicken Vegetable   Cup     USA   2.25   
3                      GGE Ramen Snack Tomato Flavor  Pack  Taiwan   2.75   
4                                    Singapore Curry  Pack   India   3.75   

  Top Ten  stars User_ID  
0     NaN   3.75   Japan  
1     NaN   1.00  Taiwan  
2     NaN   2.25     USA  
3     NaN   2.75  Taiwan  
4     NaN   3.75   India  


In [10]:
ratings_matrix = df.pivot_table(
    index='User_ID',
    columns='Brand',
    values='Stars',
    aggfunc='mean'
).fillna(0)

print("\nUser-Item Matrix Shape:", ratings_matrix.shape)


User-Item Matrix Shape: (38, 355)


In [11]:
def recommend_user_based(user_index, ratings_matrix, similarity_matrix, top_n=5):
    sim_scores = similarity_matrix[user_index]
    weighted_ratings = sim_scores @ ratings_matrix.values
    scores = weighted_ratings / (np.abs(sim_scores).sum() + 1e-9)
    return np.argsort(scores)[::-1][:top_n]

In [12]:
def recommend_item_based(user_index, ratings_matrix, similarity_matrix, top_n=5):
    user_ratings = ratings_matrix.values[user_index]
    scores = user_ratings @ similarity_matrix
    return np.argsort(scores)[::-1][:top_n]

In [13]:
def evaluate(recommended, actual, total_items):
    y_true = np.zeros(total_items)
    y_pred = np.zeros(total_items)

    y_true[actual] = 1
    y_pred[recommended] = 1

    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    return precision, recall, f1

In [14]:
# Compute similarity between users
user_similarity = cosine_similarity(ratings_matrix)

target_user = 0

recommended_users = recommend_user_based(
    target_user, ratings_matrix, user_similarity
)

print("\nRecommended Items (User-Based):")
print(ratings_matrix.columns[recommended_users].tolist())


Recommended Items (User-Based):
['Maggi', 'Suimin', 'Fantastic', 'Singa-Me', 'Trident']


In [15]:
# Compute similarity between items
item_similarity = cosine_similarity(ratings_matrix.T)

recommended_items = recommend_item_based(
    target_user, ratings_matrix, item_similarity
)

print("\nRecommended Items (Item-Based):")
print(ratings_matrix.columns[recommended_items].tolist())


Recommended Items (Item-Based):
['Trident', 'Fantastic', 'Singa-Me', 'Suimin', 'Maggi']


In [16]:
hybrid_scores = (
    ratings_matrix.values[target_user] @ item_similarity
    + user_similarity[target_user] @ ratings_matrix.values
)

hybrid_recommendations = np.argsort(hybrid_scores)[::-1][:5]

print("\nHybrid Recommendations:")
print(ratings_matrix.columns[hybrid_recommendations].tolist())


Hybrid Recommendations:
['Maggi', 'Suimin', 'Fantastic', 'Singa-Me', 'Trident']


In [17]:
# Relevant items = rating >= 4
actual_liked = np.where(ratings_matrix.values[target_user] >= 4)[0]

p_u, r_u, f_u = evaluate(recommended_users, actual_liked, ratings_matrix.shape[1])
p_i, r_i, f_i = evaluate(recommended_items, actual_liked, ratings_matrix.shape[1])
p_h, r_h, f_h = evaluate(hybrid_recommendations, actual_liked, ratings_matrix.shape[1])

print("\nUser-Based CF -> Precision:", p_u, " Recall:", r_u, " F1:", f_u)
print("Item-Based CF -> Precision:", p_i, " Recall:", r_i, " F1:", f_i)
print("Hybrid System -> Precision:", p_h, " Recall:", r_h, " F1:", f_h)


User-Based CF -> Precision: 0.2  Recall: 1.0  F1: 0.3333333333333333
Item-Based CF -> Precision: 0.2  Recall: 1.0  F1: 0.3333333333333333
Hybrid System -> Precision: 0.2  Recall: 1.0  F1: 0.3333333333333333


In [18]:
performance_data = {
    'Method': ['User-Based CF', 'Item-Based CF', 'Hybrid System'],
    'Precision': [p_u, p_i, p_h],
    'Recall': [r_u, r_i, r_h],
    'F1-Score': [f_u, f_i, f_h]
}

performance_df = pd.DataFrame(performance_data)

print("\nPerformance Comparison:")
print(performance_df)


Performance Comparison:
          Method  Precision  Recall  F1-Score
0  User-Based CF        0.2     1.0  0.333333
1  Item-Based CF        0.2     1.0  0.333333
2  Hybrid System        0.2     1.0  0.333333
